# c4fairness — worked example (binary classification)

`c4fairness` clusters the rows of a test set and then measures how a model's
**error disparities** vary across the discovered clusters and across demographic
groups. This notebook walks the full flow on a small bundled dataset using the
Python API (no R required):

**load → choose columns → encode → cluster → per-cluster error/fairness recap → heatmap.**

The dataset has all three sensitive-feature kinds: **binary** (`gender`),
**multi-categorical** (`region`), and **numeric** (`age`).

In [ ]:
import pandas as pd
from c4f.preprocessing import encode_categoricals
from c4f.clustering import cluster
from c4f.fairness_metrics import binary_error_rate_column
from c4f.cli import _build_sensitive_analysis_list, apply_salient_reconstruction
from c4f.experiments import make_recap
from c4f.result_viz import plot_cluster_recap_heatmap
from IPython.display import Image

df = pd.read_csv("data/example.csv")
df.head()

## 1. Choose columns

`regular` features drive the clustering geometry; `sensitive` features are the
protected attributes we audit. `age` is declared **continuous** so it is analysed
by its median rather than as categories.

In [ ]:
regular   = ["feat1", "feat2"]
sensitive = ["gender", "region", "age"]     # binary, multi-categorical, numeric
col_lists = {"regular": regular, "sensitive": sensitive, "proxy": [], "special": []}
orig_sensitive = list(sensitive)            # keep original names before encoding

## 2. Encode + cluster

`encode_categoricals` one-hot-encodes `region` for the Euclidean distance and
returns `multiclass_dummies` (mcd) so the tables can rebuild the readable category
later. We fix `k=3` here; pass `n_min`/`n_max` to search instead.

In [ ]:
dfe, cl, cat_names, mcd, ohe = encode_categoricals(
    df.copy(), col_lists, [], "kmeans", distance="euclidean"
)
clustering_cols = cl["regular"] + cl["sensitive"]
res = cluster(dfe[clustering_cols], algorithm="kmeans", distance="euclidean",
              n_clusters=3, random_state=42)
print("clusters:", res.n_clusters, "| silhouette:", round(res.silhouette, 3))
print("sizes:", res.cluster_sizes)

## 3. Define the error and build the recap

Binary error can be any confusion-matrix rate. Here we use the **false-positive
rate** (`fpr = FP/(FP+TN)`), derived per row from `y_true_bin` / `y_pred_bin`
(`binary_error_rate_column`). We show sensitive features in **salient** form — one
readable column per feature with its winning category — via
`_build_sensitive_analysis_list(option="salient")` + `apply_salient_reconstruction`.

In [ ]:
analysis = _build_sensitive_analysis_list(cl["sensitive"], mcd, orig_sensitive, option="salient")

dfe["fpr"] = binary_error_rate_column(df["y_true_bin"], df["y_pred_bin"], "fpr").values
res_df = dfe.copy()
res_df["clusters"] = res.labels
apply_salient_reconstruction(res_df, mcd, orig_sensitive)   # rebuild readable 'region'

recap = make_recap(res_df, clustering_cols, sensitive_cols=analysis,
                   error_col="fpr", error_type="binary",
                   feature_matrix=res.feature_matrix,
                   continuous_sensitive_cols=["age"])
recap.round(3)

Columns: `error_value` is each cluster's FPR; `error_gap` is its one-vs-all gap and
`error_gap_sig` the Fisher significance; `<feat>_value` / `<feat>_cat` describe the
sensitive make-up (winning category for `region`, median for `age`).

## 4. Heatmap

Blue = size, red = error, violet = sensitive; darker = higher (p-value columns
darker = more significant). Excel-style gridlines separate the cells.

In [ ]:
plot_cluster_recap_heatmap(recap.copy(), "binary_example", ".", error_label="FP Rate")
Image("binary_example.png")

## Takeaway

Read the `FP Rate` column down the clusters: a cluster with a markedly higher FPR
and a low `FP Rate gap sig.` p-value is where the model most over-flags — cross-
reference its `gender` / `region` / `age` columns to see which group it concentrates.
Swap `"fpr"` for `"fnr"`, `"precision"`, or `"prec_neg"` to audit a different error.